# Value-Based Modeling: Harm Aversion to fMRI — Hands-on Notebook

A short, runnable tutorial for undergraduates. We will:

1. Build a **computational model of harm aversion** (moral decision-making).
2. **Fit** it to choices and recover hidden parameters (with a recovery check).
3. Reproduce the **hyper-altruism** effect: people are more averse to harming
   *others* than *themselves*.
4. Apply the same value-modeling idea to the **real, open NARPS loss-aversion dataset** and run a
   **model-based fMRI** analysis.

> Part 1 needs only `numpy`, `scipy`, `matplotlib`. Parts 2-3 add `pandas`,
> `nilearn`, `openneuro-py`. See `README.md` for the full write-up.

## Part 1 - A computational model of harm aversion

On each trial the decider chooses between a **MORE** option (more money, more
shocks) and a **LESS** option (less money, fewer shocks). With `Δm` extra money
and `Δs` extra shocks, the subjective value of taking the deal is

$$\Delta V = (1-\kappa)\,\Delta m - \kappa\,\Delta s$$

`κ` (0-1) is **harm aversion** (0 = only money matters, 1 = won't shock for any
money). Choices follow a softmax: $P(\text{MORE}) = 1/(1+e^{-\gamma\,\Delta V})$.

In [ ]:
import numpy as np
from scipy.special import expit          # stable logistic 1/(1+exp(-x))
from scipy.optimize import minimize
from scipy import stats
import matplotlib.pyplot as plt

def choice_prob(kappa, gamma, dm, ds):
    dv = (1 - kappa) * dm - kappa * ds
    return expit(gamma * dv)

### 1.1 Simulate one participant
We generate choices from the model so we know the ground truth. Varying the
money/shock trade-off across trials is what makes `κ` identifiable.

In [ ]:
rng = np.random.default_rng(0)
n = 200
dm = rng.integers(1, 11, n).astype(float)   # extra money  1..10
ds = rng.integers(1, 11, n).astype(float)   # extra shocks 1..10

true_kappa, true_gamma = 0.6, 3.0
p = choice_prob(true_kappa, true_gamma, dm, ds)
choices = (rng.random(n) < p).astype(int)   # 1 = chose MORE
print('chose the more-money-more-shocks option on', choices.mean().round(2), 'of trials')

### 1.2 Fit the model (maximum likelihood)
Find `(κ, γ)` that make the observed choices least surprising by minimizing the
negative log-likelihood.

In [ ]:
def neg_log_lik(params, dm, ds, choices):
    kappa, gamma = params
    p = np.clip(choice_prob(kappa, gamma, dm, ds), 1e-9, 1 - 1e-9)
    return -np.sum(choices*np.log(p) + (1-choices)*np.log(1-p))

res = minimize(neg_log_lik, x0=[0.5, 1.0], args=(dm, ds, choices),
               method='L-BFGS-B', bounds=[(1e-3, 1-1e-3), (1e-3, 50)])
kappa_hat, gamma_hat = res.x
print(f'true kappa={true_kappa},  estimated kappa={kappa_hat:.2f}')
print(f'true gamma={true_gamma},  estimated gamma={gamma_hat:.2f}')

### 1.3 A whole cohort + the hyper-altruism effect
We simulate 40 participants whose true `κ_other` is drawn higher than their
`κ_self`, fit everyone, then (a) check parameter recovery and (b) test whether
people are more averse to harming others than themselves.

In [ ]:
def fit_subject(dm, ds, choices, seed=0):
    r = np.random.default_rng(seed)
    best = None
    for _ in range(8):                                  # random restarts
        x0 = [r.uniform(0.05, 0.95), r.uniform(0.1, 5)]
        res = minimize(neg_log_lik, x0, args=(dm, ds, choices),
                       method='L-BFGS-B', bounds=[(1e-3, 1-1e-3), (1e-3, 50)])
        if res.success and (best is None or res.fun < best.fun):
            best = res
    return best.x[0]                                    # kappa_hat

def trials(n, r):
    return r.integers(1, 11, n).astype(float), r.integers(1, 11, n).astype(float)

rng = np.random.default_rng(2024)
ts, eo, es = [], [], []        # true_self, est_other, est_self
to = []
for sid in range(40):
    k_self  = float(np.clip(rng.normal(0.32, 0.12), 0.02, 0.98))
    k_other = float(np.clip(rng.normal(0.55, 0.12), 0.02, 0.98))
    g = float(np.clip(rng.normal(3, 1), 0.5, 8))
    dm1, ds1 = trials(200, rng); c1 = (rng.random(200) < choice_prob(k_self,  g, dm1, ds1)).astype(int)
    dm2, ds2 = trials(200, rng); c2 = (rng.random(200) < choice_prob(k_other, g, dm2, ds2)).astype(int)
    ts.append(k_self); to.append(k_other)
    es.append(fit_subject(dm1, ds1, c1, sid)); eo.append(fit_subject(dm2, ds2, c2, sid+999))
ts, to, es, eo = map(np.array, (ts, to, es, eo))

print('recovery r (self):',  round(np.corrcoef(ts, es)[0,1], 2))
print('recovery r (other):', round(np.corrcoef(to, eo)[0,1], 2))
t, pval = stats.ttest_rel(eo, es)
print(f'kappa_self={es.mean():.2f}, kappa_other={eo.mean():.2f}')
print(f'hyper-altruism: t={t:.2f}, p={pval:.1e}, '
      f'{(eo>es).mean():.0%} of people have kappa_other>kappa_self')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].scatter(ts, es, label='self', alpha=.7); ax[0].scatter(to, eo, label='other', alpha=.7)
ax[0].plot([0,1],[0,1],'k--'); ax[0].set(xlabel='true kappa', ylabel='estimated kappa',
          title='Parameter recovery'); ax[0].legend()
ax[1].scatter(es, eo, alpha=.7); ax[1].plot([0,1],[0,1],'k--')
ax[1].set(xlabel='kappa_self', ylabel='kappa_other',
          title='Above the line = more averse to harming others')
fig.tight_layout(); plt.show()

**Read the figure.** *Left:* estimates land on the diagonal -> fitting works
(always check recovery before trusting a model). *Right:* most people sit
**above** the line (`κ_other > κ_self`) -> hyper-altruism, recovered from choices.

## Part 2 - The same idea on the real, open NARPS dataset

Harm aversion is one **value-based decision** model; **loss aversion** is
another. In **NARPS** ([ds001734](https://openneuro.org/datasets/ds001734)) 108
people accepted/rejected 50/50 gambles of a gain vs. a loss *in the scanner*.
Value of accepting: $EV = 0.5\,\text{gain} - 0.5\,\lambda\,\text{loss}$, with
`λ>1` = loss aversion. Same softmax, so we reuse the Part 1 fitting code.

Download one participant (run in a terminal):
```bash
pip install openneuro-py nilearn
python3 -m openneuro download --dataset ds001734 --target-dir ds001734 \
  --include 'sub-001/func/sub-001_task-MGT_*_events.tsv'
```

In [ ]:
# After downloading, fit loss aversion from the REAL choices (tiny files).
# This works from either the repository root or tutorials/harm-aversion-fmri/.
import os
import importlib, sys
from pathlib import Path

cwd = Path.cwd()
tutorial_dir = cwd if (cwd / 'code' / 'model_based_fmri_narps.py').exists() else cwd / 'tutorials' / 'harm-aversion-fmri'
repo_root = tutorial_dir.parents[1]
os.environ['NARPS_DIR'] = str(repo_root / 'ds001734')
sys.path.insert(0, str(tutorial_dir / 'code'))
import model_based_fmri_narps as m        # see code/ for the source
importlib.reload(m)
lam, mu = m.stage_a('sub-001')            # prints estimated lambda

## Part 3 - Model-based fMRI: where does the brain track value?

We use the model's **trial-by-trial subjective value** as a *parametric
modulator* in a first-level GLM: build a predicted time-course that rises on
high-value trials, convolve with the HRF, and find voxels that follow it.
Expected: **vmPFC / ventral striatum** for value and gain.

In [ ]:
# Needs the fMRIPrep derivatives + nilearn. Saves value/gain/loss z-maps + figures.
m.stage_b('sub-001', run=1, lam=lam)

> **Caution:** NARPS is famous because 70 teams analyzed it and disagreed
([Botvinik-Nezer 2020](https://www.nature.com/articles/s41586-020-2314-9)).
A single-subject map is a teaching artifact, not a discovery: real inference
needs the whole group and multiple-comparison correction.

### You did it
You turned a moral question into an equation, recovered its parameter from
choices, validated the method, and used the model to interrogate the brain on
real open data - the core loop of computational social neuroscience.